# AI Model Validation: From Mathematics to Trust
## Episode 12 — Reliability and Failure Analysis: How Often, Where, and Why Does an AI System Fail?

**Author: Vishwanath Akuthota**  
**Founder of DrPinnacle, OpenVals and The Foundry's**

Episode 11 established that deployed AI exists inside a changing environment. Episode 12 asks:

> **Even when average performance looks acceptable, how often does the system fail, where do failures concentrate, and how strong is our evidence about reliability?**

> **A reliable AI system is not one that never fails. It is one whose failure behaviour is measured, bounded, understood and continuously re-evaluated.**


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import platform, sys
rng=np.random.default_rng(1212)
print("Python :",sys.version.split()[0])
print("System :",platform.system(),platform.release())
print("Machine:",platform.machine())
print("NumPy  :",np.__version__)


## 1. Performance and Reliability
Performance asks how well a system performs on an evaluation. Reliability asks how consistently acceptable behaviour occurs across repeated opportunities and conditions.

### High Average Performance ≠ High Reliability


## 2. Define Failure First
Let Lᵢ be loss and L_max the maximum acceptable loss.

### Fᵢ = 1[Lᵢ > L_max]

Without an explicit failure definition, a statement such as “99% reliable” is ambiguous.


## 3. Failure Probability
For a randomly selected opportunity:

### p_fail = P(F=1)

Per-opportunity reliability:

### R = P(F=0) = 1 − p_fail

With n opportunities and k failures:

### p̂_fail = k/n

### R̂ = 1 − k/n


In [ ]:
n=5000
true_p=.02
f=rng.binomial(1,true_p,n)
k=f.sum()
print("Opportunities:",n)
print("Failures:",k)
print("Estimated p_fail:",k/n)
print("Estimated reliability:",1-k/n)


## 4. A Reliability Number Needs a Denominator

One failure in 10 opportunities and 100 failures in 100,000 opportunities are not equivalent evidence.

### Reliability Estimate + Sample Size + Sampling Process → Evidence


## 5. Binomial Reliability Model

Under simplified independent and identically distributed opportunities:

### Fᵢ ~ Bernoulli(p)

### K = ΣFᵢ ~ Binomial(n,p)

### E[p̂] = p

### Var(p̂) = p(1−p)/n

### SE(p̂) = √[p̂(1−p̂)/n]

Independence and constant failure probability are assumptions, not facts.


## 6. Wilson Interval

A point estimate is not population truth.

For observed failure proportion p̂, the Wilson interval uses:

### center = (p̂ + z²/(2n))/(1+z²/n)

### half-width = z/(1+z²/n) × √[p̂(1−p̂)/n + z²/(4n²)]


In [ ]:
def wilson(k,n,z=1.96):
    p=k/n; d=1+z*z/n
    c=(p+z*z/(2*n))/d
    h=(z/d)*np.sqrt(p*(1-p)/n+z*z/(4*n*n))
    return c-h,c+h

lo,hi=wilson(k,n)
print("Failure estimate:",k/n)
print("Approx. 95% failure interval:",(lo,hi))
print("Reliability interval:",(1-hi,1-lo))

for successes,total in [(99,100),(990,1000),(99000,100000)]:
    flo,fhi=wilson(total-successes,total)
    print(total,"trials -> reliability interval:",(1-fhi,1-flo))


## 7. Same Point Estimate, Different Evidence

99 successes in 100 and 99,000 successes in 100,000 both estimate 99% reliability.

Their uncertainty is radically different.

### Same Metric ≠ Same Evidence Strength


## 8. Zero Failures Does Not Mean Zero Risk

If k=0:

### P(K=0) = (1−p)^n

No observed failures does not logically establish p=0.


## 9. Rule of Three

With zero observed failures, an approximate 95% upper bound is:

### p_fail,upper ≈ 3/n

Therefore zero failures in 100 trials supports a very different claim from zero failures in one million trials.

### Zero Failures ≠ Perfect Reliability


In [ ]:
for n0 in [100,1000,10000,100000,1000000]:
    print(f"0 failures / {n0:>7}: approximate upper failure bound={3/n0:.6%}")


## 10. Rare-Event Evidence

From:

### p_upper ≈ 3/n

the approximate zero-failure sample requirement for target upper bound p* is:

### n ≈ 3/p*

Rare failure claims can require enormous relevant exposure.


In [ ]:
for target in [.01,.001,.0001,.00001]:
    print(f"Target {target:.5%}: approximately {np.ceil(3/target):,.0f} zero-failure trials")


## 11. Reliability Is Conditional

Global:

### P(F=1)

can hide:

### P(F=1 | G=g)

### P(F=1 | X∈A)

### P(F=1 | Stress=s)

Reliability should be evaluated under meaningful operating conditions.


In [ ]:
groups=np.array(["Nominal"]*8000+["Edge"]*1500+["Rare"]*500)
p=np.where(groups=="Nominal",.005,np.where(groups=="Edge",.04,.15))
failure=rng.binomial(1,p)

for g in ["Nominal","Edge","Rare"]:
    z=groups==g
    print(g,"N=",z.sum(),"failure rate=",failure[z].mean())
print("Aggregate failure rate:",failure.mean())


## 12. Aggregate Reliability Can Hide Fragility

Most traffic may occur in easy nominal conditions.

### Aggregate Reliability ≠ Conditional Reliability

Rare conditions can dominate consequential failure risk even when aggregate reliability looks excellent.


## 13. Failure Frequency and Severity

Let C be consequence severity.

Expected failure cost can be represented:

### E[C × F]

or across failure modes j:

### Risk = Σ_j P(F_j)C_j

This is a generic decision-theoretic representation; not every consequence should be reduced to money.

### Failure Frequency ≠ Failure Consequence


## 14. Failure Taxonomy

Useful categories may include data-quality, distribution-shift, threshold, calibration, robustness, subgroup, latency/resource, integration, human-AI interaction, and adversarial/security failures.

A taxonomy converts incidents into analyzable evidence.


## 15. Failure Clustering

If failures occur in bursts:

### P(F_t=1 | F_(t−1)=1)

may greatly exceed:

### P(F_t=1)

Simple independent Bernoulli assumptions can then understate uncertainty.


In [ ]:
T=2000
cluster=np.zeros(T,dtype=int)
state=0
for t in range(T):
    if state==0:
        if rng.random()<.01:
            state=1; cluster[t]=1
    else:
        cluster[t]=1
        if rng.random()<.35: state=0

print("Overall failure rate:",cluster.mean())
print("P(next failure | current failure):",cluster[1:][cluster[:-1]==1].mean())

plt.figure(figsize=(10,3))
plt.plot(cluster[:500],drawstyle="steps-post")
plt.xlabel("Opportunity"); plt.ylabel("Failure")
plt.title("Clustered Failures")
plt.ylim(-.1,1.1); plt.grid(alpha=.2); plt.show()


## 16. Independence Is an Assumption

Dependence can arise from shared upstream data, users, sessions, infrastructure, model versions, correlated prompts or cascading agent state.

### n Observations ≠ n Independent Pieces of Evidence

For a simplified stationary sequence with autocorrelations ρ_k:

### n_eff ≈ n / [1 + 2Σ_kρ_k]

This illustrates why correlated exposure may contain less information than raw sample count suggests.


## 17. Common-Cause Failure

Redundant components can share the same dependency.

### Redundancy ≠ Independence

If several models rely on one broken data pipeline, multiple components can fail simultaneously.


## 18. Series-System Reliability

For independent components that must all succeed:

### R_system = ΠᵢRᵢ

If k required independent components each have reliability r:

### R_system = r^k


In [ ]:
for kcomp in [1,2,5,10,20]:
    print(f"{kcomp:>2} required 99% components -> system reliability={.99**kcomp:.4%}")


## 19. Parallel Redundancy

If independent redundant components can each provide success:

### R_parallel = 1 − Πᵢ(1−Rᵢ)

Two independent 99% components would yield 99.99% under this idealized structure.

Common-cause or correlated failures can invalidate that improvement.


## 20. Fault Trees

For independent events A and B:

AND gate:

### P(A∩B) = P(A)P(B)

OR gate:

### P(A∪B) = P(A)+P(B)−P(A∩B)

Fault-tree mathematics must respect actual dependencies rather than assuming independence for convenience.


## 21. Reliability Over Time

Where time-to-failure semantics are meaningful, let T be failure time.

### R(t) = P(T>t)

### F(t) = P(T≤t)

Therefore:

### R(t) = 1−F(t)

This differs from per-prediction reliability and should only be used when the system genuinely has time-to-event semantics.


## 22. Hazard Function

For density f(t):

### h(t) = f(t)/R(t)

Hazard describes instantaneous failure tendency among systems surviving to time t.

It is not simply “probability of failure at time t.”


## 23. Exponential Reliability

If hazard is constant:

### h(t)=λ

then:

### R(t)=e^(−λt)

### F(t)=1−e^(−λt)

and:

### MTTF=1/λ

Constant hazard is a strong assumption and may be inappropriate for many AI systems.


In [ ]:
t=np.linspace(0,100,300)
for lam in [.005,.01,.03]:
    plt.plot(t,np.exp(-lam*t),label=f"λ={lam}")
plt.xlabel("Time"); plt.ylabel("R(t)")
plt.title("Exponential Reliability Functions")
plt.legend(); plt.grid(alpha=.25); plt.show()


## 24. MTTF and MTBF Need Correct Semantics

MTTF is commonly meaningful for non-repairable lifetime systems.

MTBF is commonly used for repairable systems.

Prediction models, APIs, agent workflows and physical AI systems are different reliability objects.

### Reliability Metric Must Match System Semantics


## 25. Multi-Step Agent Reliability

If every one of k independent required steps succeeds with probability r:

### R_workflow = r^k

Even excellent step-level reliability can compound into much weaker end-to-end reliability.


In [ ]:
r=.99
steps=np.arange(1,51)
workflow=r**steps

plt.figure(figsize=(8,5))
plt.plot(steps,workflow)
plt.xlabel("Required workflow steps")
plt.ylabel("End-to-end reliability")
plt.title("Reliability Compounds Across Required Steps")
plt.grid(alpha=.25); plt.show()

print("99% per step, 20 steps:",r**20)


For r=0.99 and k=20:

### R_workflow ≈ 81.8%

This is one reason agentic AI must be evaluated end-to-end rather than only tool-by-tool.

The formula assumes independent identical required steps; real workflows may include correlation, branching, retries and recovery.


## 26. Retry Logic

If one attempt succeeds with probability r and attempts are independent:

### P(success within m attempts) = 1−(1−r)^m

Retries can improve completion probability while increasing latency, cost, duplicate-action risk or downstream side effects.


In [ ]:
r=.85
for m in [1,2,3,5]:
    print(f"{m} attempt(s): {1-(1-r)**m:.4%}")


## 27. Multidimensional Reliability

A system can fail because of quality, latency, safety or infrastructure.

### F_total = F_quality ∪ F_latency ∪ F_safety ∪ F_system

Reliability requirements should reflect the intended service, not merely predictive accuracy.


## 28. Reliability Under Drift

Episode 11 established that deployment distributions change.

Therefore:

### p_fail(t) = P_t(F=1)

A reliability estimate measured at t₀ does not automatically remain valid at t₁.

### Reliability Is Contextual and Temporal


## 29. Reliability Growth and Regression

Across versions v:

### p_fail(v)

can be monitored to test whether remediation actually reduces failures.

A new version can improve average performance while reintroducing historically important failure modes.

### New Version Better on Average ≠ No Reliability Regression


In [ ]:
versions=np.arange(1,9)
rates=np.array([.08,.065,.061,.045,.043,.031,.028,.019])
plt.figure(figsize=(8,5))
plt.plot(versions,rates,marker="o")
plt.xlabel("System version"); plt.ylabel("Observed failure rate")
plt.title("Illustrative Reliability Growth")
plt.grid(alpha=.25); plt.show()


## 30. Failure Reproduction

For material failures preserve model/version, input, context/state, output, expected behaviour, category, severity, timestamp, dependencies, environment and recovery behaviour.

A failure that cannot be reconstructed is difficult to investigate scientifically.


## 31. Root Cause and Contributing Conditions

AI incidents may involve:

### Root Cause
### Contributing Conditions
### Triggering Event
### Detection Failure
### Recovery Failure

The observed failure may result from model error, bad input, threshold policy and integration state simultaneously.


## 32. Reliability, Robustness, Safety and Availability

### Reliability ≠ Robustness ≠ Safety ≠ Availability

Reliability: consistency of acceptable behaviour.

Robustness: stability under perturbation or shift.

Safety: avoidance/control of unacceptable harm.

Availability: whether the service is operational.

These dimensions interact but should not be substituted for one another.


## 33. Reliability Monitoring

A deployed system can monitor:

### p̂_fail(t)

### p̂_fail(t | subgroup)

### p̂_fail(t | scenario)

### Severity(t)

### Recovery Rate(t)

### Recurrence Rate(t)

### Time-to-Detection(t)

### Time-to-Recovery(t)

This creates a failure evidence stream rather than a static percentage.


## 34. Monitoring Boundaries

For baseline failure rate p₀ and independent Bernoulli window size n:

### SE₀ = √[p₀(1−p₀)/n]

An illustrative upper boundary might be:

### p₀ + zSE₀

A boundary crossing is an investigation trigger, not proof that the system became unsafe.


In [ ]:
p0=.02
window=1000
se=np.sqrt(p0*(1-p0)/window)
for z in [2,3]:
    print(f"{z}-SE upper boundary:",p0+z*se)


## 35. Sequential Monitoring and False Alarms

Repeated monitoring creates repeated opportunities for random threshold crossings.

Alert design may therefore require persistence, severity tiers, sequential methods or multiple-testing awareness.

### More Monitoring ≠ Automatically Better Evidence


## 36. Reliability Evidence Hierarchy

A defensible reliability claim combines:

### Failure Definition
### Representative Exposure
### Sample Size
### Conditional Analysis
### Uncertainty
### Failure Severity
### Dependency Analysis
### Root-Cause Evidence
### Temporal Monitoring

A single success percentage is only one layer.


## 37. OpenVals Perspective

A serious validation platform should preserve:

- reliability claim and unit of opportunity
- failure definition and acceptance threshold
- total exposure and failure count
- failure probability and confidence interval
- zero-failure upper bounds
- subgroup and scenario failure rates
- failure severity and taxonomy
- clustering/dependence indicators
- common-cause dependencies
- end-to-end workflow reliability
- retries and recovery
- model/version
- regression suite
- root-cause investigation
- reliability over time
- validation trigger
- post-remediation evidence

The objective is not to advertise “five nines” without context.

It is to preserve the evidence required to understand exactly what a reliability claim means.


## 38. Scientific Reliability Chain

### Opportunity → Failure Definition → Exposure → Failure Observation → Probability → Uncertainty → Conditional Analysis → Root Cause → Remediation → Revalidation

This transforms reliability from a marketing percentage into an empirical evidence process.


## 39. Mathematical Summary

### Fᵢ = 1[Lᵢ>L_max]

### p_fail = P(F=1)

### R = 1−p_fail

### p̂_fail = k/n

### K ~ Binomial(n,p)

### SE(p̂)=√[p̂(1−p̂)/n]

### P(K=0)=(1−p)^n

### p_upper ≈ 3/n  for zero observed failures at approximately 95% confidence

### P(F=1|G=g)

### R_series=ΠᵢRᵢ

### R_parallel=1−Πᵢ(1−Rᵢ)

### R(t)=P(T>t)

### F(t)=1−R(t)

### h(t)=f(t)/R(t)

### R_exponential(t)=e^(−λt)

### MTTF=1/λ  under exponential lifetime assumptions

### R_workflow=r^k  under independent identical required-step assumptions

### P(success within m attempts)=1−(1−r)^m

### Risk=Σ_jP(F_j)C_j


## 40. Central Conclusion

### High Success Rate ≠ Complete Reliability Evidence

> **Reliability is not the absence of observed failure. It is quantified evidence about the probability, conditions, severity, dependence and recurrence of failure.**

Zero observed failures can still contain substantial uncertainty.

Excellent aggregate reliability can hide rare-condition fragility.

Multi-step agents can compound failure risk.

Redundant systems can share common causes.

Repeated failures can violate independence assumptions.

A reliability percentage without a denominator, uncertainty and failure definition is incomplete evidence.

The scientific questions are:

### How often does the system fail?
### Where does it fail?
### How severe are those failures?
### Are failures independent or correlated?
### How strong is the evidence?
### Does reliability persist over time?
### What changed after remediation?

> **Trust requires not only measuring success, but scientifically characterizing failure.**

## Next Episode

**Episode 13 — Safety, Risk, and Harm: When Model Error Becomes Consequence**

Episode 13 can connect failure probability to consequence severity through expected risk, hazard identification, loss distributions, tail risk, safety constraints, human oversight, fail-safe behaviour, residual risk and AI assurance.
